### Setting the topology

In a YAML file, define the parameters and topology for your test in a similar manner as the following:
```yaml
topology:
  name: "g5k_mcast_eval"
  wall_time: "2hr"
  relay_nodes: false # whether to add one relay machine in each cluster
  netns_per_client: 5 # number of network namespaces to run on each client
  # see the possible frrouting version at https://deb.frrouting.org/
  frrouting_version: "frr-10.4"
  router_template: "base_router_config_ospf.frr" # path to the router configuration template

  server:
    cluster: "chirop" # Lille
    nodes: 1
    # node: "chirop-5.lille.grid5000.fr"   # optional: pin a specific machine

  # each site has one router + num_clients clients and one relay,
  # all reserved in the given cluster. 
  sites:
    - name: nancy
      cluster: gros
      num_clients: 5
    - name: rennes
      cluster: parasilo
      num_clients: 5
    - name: nantes
      cluster: ecotype
      num_clients: 5
    - name: lyon
      cluster: nova
      num_clients: 5

  # links are established between routers in different clusters
  # GRE tunnels are established between the two routers, with OSPF running over it.
  # endpoints must be router_server or router_client_CLIENT-CLUSTER-ID
  links:
    - [router_server, router_client_0]             # src -> nancy
    - [router_client_0, router_client_1]           # nancy -> rennes
    - [router_client_0, router_client_3]           # nancy -> lyon
    - [router_client_0, router_client_2]           # rennes -> nantes
``` 


In [ ]:
# !pip install enoslib ipywidgets==8.1.5 fabric --break-system-packages
!pip install -U jupyterlab ipywidgets jupyterlab-widgets --break-system-packages


### Setting up the experiment
Once you have your `topology.yaml` file, you can create the `G5KExpe` class which will handle most things for you.

In [37]:
from g5k_eval import G5KExpe

experiment = G5KExpe(
    # change the path to point to your topology yaml file
    topology_conf="./mcast_eval.yaml",
    #
    # other parameters exist:
    # g5k_conf_file_loc points to your .python-grid5000.yaml file which contains your grid5000 credentials, by default it is in `~/` (so `/home/USERNAME`)
    # g5k_conf_file_loc=".python-grid5000.yaml"
    #
    # job_type should be deploy, but you may need it to be different
    # job_type="deploy"
    #
    # os_env_name defines the OS environement that is deployed on the machines
    # by default it is debian12 with NFS, however you can find the entire list at https://www.grid5000.fr/w/Getting_Started#:~:text=On%20Grid%275000%20reference%20environments
    # Make sure to pick debian to ensure that the packages are properly installed
    # os_env_name="debian12-nfs"
    #
    # You can configure the number of ansible forks used, ansible's default is 5, meaning that it'll run commands on at most 5 host at once
    # in this framework the default is 25 to make use of more parallelism, however, increasing this value will consume more resources (especially memory)
    # setting the number of forks to 200 will consume around 25 GB of memory but will allow ansible to perform operations on 200 hosts at the same time
    ansible_forks=20,
)

# you should always follow grid5000's usage policy (see https://www.grid5000.fr/w/Grid5000:UsagePolicy)
# this method simply checks that the job you are trying to start will not cross the day-night boundary.
# If it does, it'll warn you. You can always comment this out if you wish...
experiment.usage_policy_check()

provider = experiment.setup_enoslib_conf()

_____        ___  ____  _ _ _
 | ____|_ __  / _ \/ ___|| (_) |__
 |  _| | '_ \| | | \___ \| | | '_ \
 | |___| | | | |_| |___) | | | |_) |
 |_____|_| |_|\___/|____/|_|_|_.__/  10.9.0

 • Documentation: ]8;id=497077;https://discovery.gitlabpages.inria.fr/enoslib/\https://discovery.gitlabpages.inria.fr/enoslib/]8;;\                            
 • Source: ]8;id=232308;https://gitlab.inria.fr/discovery/enoslib\https://gitlab.inria.fr/discovery/enoslib]8;;\                                         
 • Chat: ]8;id=947406;https://framateam.org/enoslib\https://framateam.org/enoslib]8;;\

                         Dependency check                         
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider      ┃    Status     ┃ Hint                           ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Chameleon     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonKVM  │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonEdge │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Fabric        │ NOT INSTALLED │ pip install enoslib[fabric]    │
│ Distem        │ NOT INSTALLED │ pip install enoslib[distem]    │
│ IOT-lab       │ NOT INSTALLED │ pip install enoslib[iotlab]    │
│ Grid'5000     │   INSTALLED   │                                │
│ Openstack     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Vagrant       │ NOT INSTALLED │ pip install enoslib[vagrant]   │
│ VMonG5k       │   INSTALLED   │                                │
└───────────────┴───────────────┴────────────────────────────────┘

                                Connectivity check                                 
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider  ┃ Key                 ┃ Connectivity ┃ Hint                           ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Grid'5000 │ ssh:access          │      ✅      │ Connection to access.grid5000… │
│ Grid'5000 │ ssh:access:frontend │      ✅      │ Connection Host(rennes.grid50… │
│ Grid'5000 │ api:access          │      ✅      │                                │
│ VMonG5k   │ access              │      ❔      │ Check G5k status               │
└───────────┴─────────────────────┴──────────────┴────────────────────────────────┘


### Reserving resources
Now that G5K is setup, we can create the experiment's reservation by defining the number of machines of each role and in each cluster.

Once done, we proceed with the actual reservation of the machines. Be aware that this step may take some time (minimum 5 minutes). This is due to the deployment of the VM image. 

**Don't forget to run "ssh-add KEY_PATH" to allow ansible to connect using your ssh key**

In [38]:
!pip install -U jupyterlab ipywidgets jupyterlab-widgets --break-system-packages
experiment.reserve_res(provider)
display(experiment.roles)

Defaulting to user installation because normal site-packages is not writeable
Reserving resources now, might take a while...


INFO     [ProviderS] Common reservation_date=2026-09-16T16:03:57 (local time) ]8;id=428415;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/providers.py\providers.py]8;;\:]8;id=302346;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/providers.py#60\60]8;;\
         [5 providers]                                                                       

INFO     [G5k] Submitting {'name': 'g5k_mcast_eval', 'types': ['deploy', ]8;id=777637;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=921149;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#306\306]8;;\
         'origin=enoslib_g5k'], 'resources': "{cluster='parasilo'}/nodes                     
         =1+{cluster='parasilo'}/nodes=5+slash_22=1,walltime=2:00:00",                       
         'command': 'sleep 31536000', 'queue': 'default', 'reservation':                     
         '2026-09-16 16:04:00'} on rennes                                                    

INFO     [G5k] Submitting {'name': 'g5k_mcast_eval', 'types': ['deploy', ]8;id=54133;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=197132;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#306\306]8;;\
         'origin=enoslib_g5k'], 'resources': "{cluster='gros'}/nodes=1+{                     
         cluster='gros'}/nodes=5+slash_22=1,walltime=2:00:00",                               
         'command': 'sleep 31536000', 'queue': 'default', 'reservation':                     
         '2026-09-16 16:05:09'} on nancy                                                     

INFO     [G5k] Submitting {'name': 'g5k_mcast_eval', 'types': ['deploy', ]8;id=853089;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=284147;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#306\306]8;;\
         'origin=enoslib_g5k'], 'resources': "{cluster='nova'}/nodes=1+{                     
         cluster='nova'}/nodes=5+slash_22=1,walltime=2:00:00",                               
         'command': 'sleep 31536000', 'queue': 'default', 'reservation':                     
         '2026-09-16 16:06:35'} on lyon                                                      

...

Output()

Results : []


Finished 5 tasks (Gather facts,Ensure apt keyring directory exists,Download FRR GPG key,Add 
FRR apt repository,Install FRR packages) on {'chirop-1.lille.grid5000.fr', 
'ecotype-23.nantes.grid5000.fr', 'parasilo-1.rennes.grid5000.fr', 'nova-1.lyon.grid5000.fr', 
'gros-105.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

ip
127.0.0.1/8 # noqa
::1/128 # noqa
ip
fe80::9a03:9bff:feb0:c45e/64 # noqa
172.16.66.105/20 # noqa
ip
127.0.0.1/8 # noqa
::1/128 # noqa
ip
fe80::262:bff:fea7:58b2/64 # noqa


**Optional**: You can refresh the roles by running the cell below (useful when the notebook closed but you have a reservation running)

In [ ]:
experiment.sync_info()

#### Setting up interfaces, IP subnets, and Network namespaces

In [39]:
experiment.setup_interfaces()
experiment.assign_node_ips()
experiment.netns_setup_macvlan()

Output()

Prod interface for nova-4.lyon.grid5000.fr: enp5s0f0
Prod interface for chirop-5.lille.grid5000.fr: ens10f0np0
Prod interface for chirop-1.lille.grid5000.fr: ens10f0np0
Prod interface for parasilo-19.rennes.grid5000.fr: eno1
Prod interface for gros-111.nancy.grid5000.fr: eno1
Prod interface for ecotype-34.nantes.grid5000.fr: eno1
Prod interface for ecotype-23.nantes.grid5000.fr: eno1
Prod interface for gros-109.nancy.grid5000.fr: eno1
Prod interface for parasilo-28.rennes.grid5000.fr: eno1
Prod interface for ecotype-46.nantes.grid5000.fr: eno1
Prod interface for parasilo-1.rennes.grid5000.fr: eno1
Prod interface for nova-1.lyon.grid5000.fr: enp5s0f0
Prod interface for nova-5.lyon.grid5000.fr: enp5s0f0
Prod interface for ecotype-45.nantes.grid5000.fr: eno1
Prod interface for ecotype-40.nantes.grid5000.fr: eno1
Prod interface for gros-105.nancy.grid5000.fr: eno1
Prod interface for nova-8.lyon.grid5000.fr: enp5s0f0
Prod interface for parasilo-18.rennes.grid5000.fr: eno1
Prod interface for

Finished 1 tasks (cmd) on {'chirop-5.lille.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

...

Output()

Created 25 namespaces for client_2 (with gateway 10.176.0.1)
gateway_ip=10.140.0.1 for client client_3


Finished 1 tasks (create_macvlan_namespaces) on {'nova-5.lyon.grid5000.fr', 
'nova-23.lyon.grid5000.fr', 'nova-10.lyon.grid5000.fr', 'nova-4.lyon.grid5000.fr', 
'nova-8.lyon.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Created 25 namespaces for client_3 (with gateway 10.140.0.1)


### Setting up GRE tunnels between routers in different clusters

This step will create GRE tunnels between each pair of routers as defined in the topology file. The endpoints of the tunnels use the production IP of the nodes.

In [40]:
experiment.setup_gre_tunnels()

Output()

role_a: router_server
role_b: router_client_0
self.roles[role_a]: {Host(address='chirop-1.lille.grid5000.fr', alias='chirop-1.lille.grid5000.fr', user='root', keyfile=None, port=None, extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry', 'ips': ['10.136.0.1', '172.16.33.1']}, net_devices={NetDevice(name='lo', addresses={IPAddress(network=None, ip=IPv4Interface('127.0.0.1/8')), IPAddress(network=None, ip=IPv6Interface('::1/128'))}), NetDevice(name='ens10f0np0', addresses={IPAddress(network=None, ip=IPv6Interface('fe80::262:bff:fea7:58b2/64')), IPAddress(network=<enoslib.infra.enos_g5k.objects.G5kEnosProd4Network object at 0x71e29fb5bd10>, ip=IPv4Interface('172.16.33.1/20'))}), NetDevice(name='ens10f1np1', addresses=set())}, _Host__original_extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'})}
self.roles[role_b]: {Host(address='gros-105.nancy.grid5000.fr', alias='gros-105.nancy.grid5000.fr', user='root', keyfile=None, port=None, extra={'gateway': 'access.grid500

Finished 1 tasks (setup_gre_router_server) on {'chirop-1.lille.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

...

Output()

Created 1 GRE tunnels on router_client_3 (nova-1.lyon.grid5000.fr)


Finished 1 tasks (setup_gre_router_client_2) on {'ecotype-23.nantes.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Created 1 GRE tunnels on router_client_2 (ecotype-23.nantes.grid5000.fr)
Router tunnels: {'router_server': [{'iface': 'gre1', 'ip': '192.168.0.1', 'network': '192.168.0.0', 'tunnel_subnet': IPv4Network('192.168.0.0/30')}], 'router_client_0': [{'iface': 'gre1', 'ip': '192.168.0.2', 'network': '192.168.0.0', 'tunnel_subnet': IPv4Network('192.168.0.0/30')}, {'iface': 'gre2', 'ip': '192.168.0.5', 'network': '192.168.0.4', 'tunnel_subnet': IPv4Network('192.168.0.4/30')}, {'iface': 'gre3', 'ip': '192.168.0.9', 'network': '192.168.0.8', 'tunnel_subnet': IPv4Network('192.168.0.8/30')}], 'router_client_1': [{'iface': 'gre1', 'ip': '192.168.0.6', 'network': '192.168.0.4', 'tunnel_subnet': IPv4Network('192.168.0.4/30')}, {'iface': 'gre2', 'ip': '192.168.0.13', 'network': '192.168.0.12', 'tunnel_subnet': IPv4Network('192.168.0.12/30')}], 'router_client_3': [{'iface': 'gre1', 'ip': '192.168.0.10', 'network': '192.168.0.8', 'tunnel_subnet': IPv4Network('192.168.0.8/30')}], 'router_client_2': [{'ifac

#### FRRouting setup

With GRE tunnels setup between routers, we can now configure and start FRRouting. The frr configuration template defined in the topology file will be used as a base.

In [41]:
experiment.frrouting_setup()
experiment.setup_default_routes()

Output()

Finished 1 tasks (restart_frr_chirop-1.lille.grid5000.fr) on {'chirop-1.lille.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

[router_server] chirop-1.lille.grid5000.fr  prod=10.136.0.1  loopback=10.136.3.254  gateway=172.16.47.254  tunnels=1


Output()

...

[router_client_2] ecotype-23.nantes.grid5000.fr  prod=10.176.0.1  loopback=10.176.3.254  gateway=172.16.207.254  tunnels=1


Output()

Finished 1 tasks (restart_frr_nova-1.lyon.grid5000.fr) on {'nova-1.lyon.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

[router_client_3] nova-1.lyon.grid5000.fr  prod=10.140.0.1  loopback=10.140.3.254  gateway=172.16.63.254  tunnels=1
172.16.33.1
172.16.66.105
Unknown role: relay_0
172.16.97.1
Unknown role: relay_1
172.16.193.23
Unknown role: relay_2
172.16.52.1
Unknown role: relay_3
setting default routes on 21 nodes
default via 172.16.47.254 dev ens10f0np0
chirop-5.lille.grid5000.fr's default route is 172.16.33.1 on ens10f0np0
default via 172.16.79.254 dev eno1
gros-109.nancy.grid5000.fr's default route is 172.16.66.105 on eno1
default via 172.16.79.254 dev eno1
gros-110.nancy.grid5000.fr's default route is 172.16.66.105 on eno1
default via 172.16.79.254 dev eno1
gros-11.nancy.grid5000.fr's default route is 172.16.66.105 on eno1
default via 172.16.79.254 dev eno1
gros-111.nancy.grid5000.fr's default route is 172.16.66.105 on eno1
default via 172.16.79.254 dev eno1
gros-113.nancy.grid5000.fr's default route is 172.16.66.105 on eno1
default via 172.16.111.254 dev eno1
parasilo-19.rennes.grid5000.fr's d

### Upload binary files over to nodes

We build the executables locally first

In [42]:
# TODO: change this path to your project
!cd ../../../g5k_mcast_eval && cargo build --release

   --> /home/corentin/fcquic_applications_master_thesis/multicast-quic/octets/src/lib.rs:474:22
    |
474 |     pub fn get_bytes(&mut self, len: usize) -> Result<Octets> {
    |                      ^^^^^^^^^                        ^^^^^^ the same lifetime is hidden here
    |                      |
    |                      the lifetime is elided here
    |
    = help: the same lifetime is referred to in inconsistent ways, making the signature confusing
    = note: `#[warn(mismatched_lifetime_syntaxes)]` on by default
help: use `'_` for type paths
    |
474 |     pub fn get_bytes(&mut self, len: usize) -> Result<Octets<'_>> {
    |                                                             ++++

   --> /home/corentin/fcquic_applications_master_thesis/multicast-quic/octets/src/lib.rs:491:26
    |
491 |     pub fn get_bytes_mut(&mut self, len: usize) -> Result<OctetsMut> {
    |                          ^^^^^^^^^                        ^^^^^^^^^ the same lifetime is hidden here
    | 

Then we push them to the nodes

In [43]:
import enoslib as en

experiment.push_binaries(
    # TODO: change these paths with the path to your binaries and certificates
    bin_dir="../../../g5k_mcast_eval/target/release",
    cert_dir="../../../g5k_mcast_eval",
    relay_binaries=(),
)

res = en.run_command(
    "sysctl -w net.core.rmem_default=26214400 && sysctl -w net.core.rmem_max=26214400",
    roles=experiment.roles,
)
print("errors: " + str([out.stderr for out in res.filter(status=en.STATUS_FAILED)]))

Output()

Finished 2 tasks (file,copy) on {'ecotype-45.nantes.grid5000.fr', 
'ecotype-34.nantes.grid5000.fr', 'ecotype-47.nantes.grid5000.fr', 
'gros-110.nancy.grid5000.fr', 'parasilo-19.rennes.grid5000.fr', 'gros-11.nancy.grid5000.fr', 
'ecotype-40.nantes.grid5000.fr', 'nova-4.lyon.grid5000.fr', 'gros-109.nancy.grid5000.fr', 
'nova-5.lyon.grid5000.fr', 'nova-23.lyon.grid5000.fr', 'parasilo-18.rennes.grid5000.fr', 
'nova-10.lyon.grid5000.fr', 'parasilo-28.rennes.grid5000.fr', 
'parasilo-10.rennes.grid5000.fr', 'nova-8.lyon.grid5000.fr', 'gros-113.nancy.grid5000.fr', 
'gros-111.nancy.grid5000.fr', 'parasilo-14.rennes.grid5000.fr', 'chirop-5.lille.grid5000.fr',
'ecotype-46.nantes.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Pushed ['server', 'client'] to 21 server/client node(s)


Finished 1 tasks (sysctl -w net.core.rmem_default=26214400 && sysctl -w 
net.core.rmem_max=26214400) on {'ecotype-45.nantes.grid5000.fr', 
'ecotype-34.nantes.grid5000.fr', 'ecotype-47.nantes.grid5000.fr', 
'gros-110.nancy.grid5000.fr', 'nova-1.lyon.grid5000.fr', 'gros-105.nancy.grid5000.fr', 
'parasilo-19.rennes.grid5000.fr', 'gros-11.nancy.grid5000.fr', 
'ecotype-40.nantes.grid5000.fr', 'nova-4.lyon.grid5000.fr', 'gros-109.nancy.grid5000.fr', 
'nova-5.lyon.grid5000.fr', 'parasilo-18.rennes.grid5000.fr', 'nova-23.lyon.grid5000.fr', 
'nova-10.lyon.grid5000.fr', 'parasilo-28.rennes.grid5000.fr', 
'parasilo-10.rennes.grid5000.fr', 'nova-8.lyon.grid5000.fr', 'chirop-1.lille.grid5000.fr', 
'gros-113.nancy.grid5000.fr', 'parasilo-1.rennes.grid5000.fr', 'gros-111.nancy.grid5000.fr', 
'ecotype-23.nantes.grid5000.fr', 'parasilo-14.rennes.grid5000.fr', 
'chirop-5.lille.grid5000.fr', 'ecotype-46.nantes.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

errors: []


### Running the relay experiment

Experiment.py provides some basic blocks that should (ideally) allow you to define your own custom experiments.
Below you will find the code for the evaluation of two types of Flexicast QUIC relays, this should hopefully provide enough information.


In [44]:
import concurrent.futures
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Literal
import time

import enoslib as en

from g5k_eval.experiment import (
    EvalConfig,
    MetricSpec,
    collect_results,
    run_eval,
)
from g5k_eval.remote import (
    run_cmd_bg_enos,
    run_cmd_ssh_parallel,
    send_pkill_hosts,
    ssh_bg_hosts,
)


@dataclass
class RunConfig:
    additional_data_size: int
    test_length: int


@dataclass
class CatEvalConfig(EvalConfig):
    ready_sleep_clients: int = 2
    post_test_buffer: int = 7
    bin_log_level: str = "info"
    cert_path: str = "/tmp"
    server_bin: str = "/tmp/bin/server"
    client_bin: str = "/tmp/bin/client"
    remote_log_root: str = "/tmp/logs"
    num_ns_per_client: int = experiment.topology.netns_per_client
    cc_algo: str = "cubic"
    fallback_delay: int = 10000
    server_cpus: str = "0-7"  # taskset -c range for server
    per_cluster_results: bool = True
    flow_control: int = 16_000_000  # 16 Megabytes


# define the results to extract from the client logs, one csv output file is emitted for each metric
METRICS = [
    MetricSpec(
        key="LATENCY",
        column="y_LATENCY",
        pattern=rf"^RESULT-LATENCY-\S+\s+([0-9.]+)\s*$",
    ),
    #  you can add more result types here, e.g.:
    # MetricSpec(key="THROUGHPUT", column="y_THROUGHPUT"),
]

We can now define specific tests based on our `RunConfig`.

For the network categorization test, we simply send one packet of increasing size, and we wait until it has been received by all clients.
- 1KB will fit inside of one packet
- 10KB will fit in one flight of packets
- for larger values, the sender will have to grow its congestion window to be able to send it

In [46]:
def categorization_matrix():
    return [
        RunConfig(additional_data_size=sz, test_length=length)
        # for sz in (10_000, 100_000, 1_000_000, 10_000_000)
        # for sz in (1_000,)
        # for sz in (1_000_000,)
        # for sz in (10_000_000,)
        # for sz, length in ((10_000_000, 20),)
        for sz, length in (
            # (1_000, 10),
            (10_000, 10),
            (100_000, 15),
            (1_000_000, 20),
            (10_000_000, 20),
        )
    ]

To enable us to have graphs that show certain metrics per cluster, we need to pass in a list of the cluster names and their subnets to the clients. Here we construct the lists to pass to the clients

In [47]:
# ---------- cluster names & subnets ----------
# the subnets are in experiment.networks, but i need the subnets keyed by their index and not their cluster
site_subnets = {
    site.name: {
        "cluster": site.cluster,
        "num_clients": site.num_clients,
        "subnet": experiment.networks[f"subnet_client_{i}"][
            0
        ],  # NOTE: we only remove the prefix because the arg is an IPv4Addr (in rust), not a network  # "10.x.y.0"
    }
    for i, site in enumerate(experiment.topology.sites)
}
server_subnet = str(experiment.networks["subnet_server"][0].network)

cluster_names = [site.name for site in experiment.topology.sites]
cluster_subnets = [
    str(info["subnet"].network.network_address) for info in site_subnets.values()
]

site_subnets_str = [str(info["subnet"].network) for info in site_subnets.values()]
print(f"site_subnets_str: {site_subnets_str}")
print(f"cluster_names:  {cluster_names}")
print(f"cluster_subnets: {cluster_subnets}")

site_subnets_str: ['10.144.36.0/22', '10.158.4.0/22', '10.176.0.0/22', '10.140.0.0/22']
cluster_names:  ['nancy', 'rennes', 'nantes', 'lyon']
cluster_subnets: ['10.144.36.0', '10.158.4.0', '10.176.0.0', '10.140.0.0']


Now that the experiment is defined, we need to specify the commands that will be ran on the nodes.


In [48]:
# ---------- command builders ----------
def server_cmd(cfg, rc, server_ip, run_dir, sleep_deadline_ts):
    length = rc.test_length * 2
    qlog = f"{run_dir}/qlog/server"
    return (
        f"mkdir -p {qlog} && "
        f"env QLOGDIR={qlog} RUST_LOG_STYLE=never RUST_BACKTRACE=full "
        f"RUST_LOG={cfg.bin_log_level} taskset -c {cfg.server_cpus} {cfg.server_bin} "
        f"--cert-path {cfg.cert_path} --src {server_ip}:4433 --mc-src-addr {server_ip}:4443 "
        f"--flexicast --fc-timer 0 --fall-back-delay {cfg.fallback_delay} "
        f"--unicast --fec-scheduler noredundancy --length {length} "
        f"--cc-algorithm {cfg.cc_algo} --fc-cwnd {cfg.cc_algo} "
        f"--additional-data-size {rc.additional_data_size} --test-start-ts {sleep_deadline_ts} "
        f"--initial-fc-flow {cfg.flow_control} "
    )


# IMPORTANT NOTE: since we have multiple network namespaces defined on each client machine,
# we can run processes in these namespaces using the naming scheme "client-$NS_IDX" (with NS_IDX going from the number of 0 to NSs)
def client_loop_cmd(cfg, rc, server_ip, run_dir, node_id, sleep_deadline_ts):
    qlog_base = f"{run_dir}/qlog/client"
    per_cluster_res = "--per-cluster-results" if cfg.per_cluster_results else ""
    return f"""
mkdir -p {run_dir}/client
pids=()
for NS_IDX in $(seq $(( {cfg.num_ns_per_client} - 1 )) -1 0); do
    GLOBAL_IDX=$(( {node_id} * {cfg.num_ns_per_client} + NS_IDX ))
    CLIENT_ID=$(( GLOBAL_IDX + 1 ))
    NS_NAME="client-$NS_IDX"
    mkdir -p {qlog_base}_$CLIENT_ID
    CLIENT_IP=$(ip netns exec $NS_NAME ip -f inet addr show | grep inet | tail -1 | awk '{{print $2}}' | cut -d'/' -f1)
    ip netns exec $NS_NAME env QLOGDIR={qlog_base}_$CLIENT_ID RUST_LOG_STYLE=never RUST_BACKTRACE=full RUST_LOG={cfg.bin_log_level} \\
        {cfg.client_bin} --server-ip {server_ip} --port 4433 \\
        -l $CLIENT_IP --flexicast -u CLIENT$CLIENT_ID --length {rc.test_length} \\
        --test-start-ts {sleep_deadline_ts} \\
        --additional-data-size {rc.additional_data_size} --cc-algorithm {cfg.cc_algo} --flow-control {cfg.flow_control}  \\
        {per_cluster_res} \\
         {" ".join(f"--cluster-names={name}" for name in cluster_names)} \\
         {" ".join(f"--cluster-subnets={subnet}" for subnet in cluster_subnets)} \\
        > {run_dir}/client/client_$CLIENT_ID.stdout \\
        2> {run_dir}/client/client_$CLIENT_ID.stderr < /dev/null < /dev/null &
    pids+=($!)
done
for pid in "${{pids[@]}}"; do wait $pid; done
"""


# ---------- one run of the relay experiment ----------
def run_once(cfg, rc, run_index, test_name):
    """Run one iteration: start the server, start the clients in
    their namespaces, wait for the test to finish, then collect the results."""
    roles_dict = experiment.roles
    node_ips = experiment.node_ips

    server_ip = node_ips["server"][0]
    # NOTE: very important, make sure that this run_id is the same as the one in the SQLOG collection cell below
    run_id = f"sz{rc.additional_data_size}_r{run_index}"
    run_dir = f"{cfg.remote_log_root}/{test_name}/{run_id}"

    # make sure that each client is root because it has to start the clients in network namespaces
    client_hosts = [
        en.Host(h.address, alias=h.alias, user="root", extra=h.extra)
        for h in roles_dict["client"]
    ]
    all_hosts = roles_dict["server"] + client_hosts

    # create the dirs on all of the hosts and stop anything left over from a
    # previous run
    run_cmd_ssh_parallel(
        f"mkdir -p {run_dir}/server {run_dir}/client {run_dir}/qlog ; "
        f"pkill -9 server || true ; pkill -9 client || true",
        all_hosts,
    )
    time.sleep(1)

    # pick a timestamp in 5 seconds, we pass this to all of the clients that will all wait until that timestamp is reached before starting
    datetime_now = datetime.now()
    sleep_deadline = datetime_now + timedelta(seconds=5)
    sleep_deadline_ts = sleep_deadline.timestamp()

    # start server in bg
    run_cmd_bg_enos(
        server_cmd(cfg, rc, server_ip, run_dir, sleep_deadline_ts),
        roles_dict["server"],
        stdout=f"{run_dir}/server/server.stdout",
        stderr=f"{run_dir}/server/server.stderr",
        task_name="server",
    )

    # start all clients at once, in a single ansible run: each host gets its own
    # command, and they all wait for sleep_deadline_ts so they start together
    ssh_bg_hosts(
        [
            (
                h,
                client_loop_cmd(
                    cfg, rc, server_ip, run_dir, node_id, sleep_deadline_ts
                ),
                f"{run_dir}/client/loop_{node_id}.stdout",
                f"{run_dir}/client/loop_{node_id}.stderr",
            )
            for node_id, h in enumerate(client_hosts)
        ],
        task_name="clients",
    )
    print("Started clients")

    # wait for test duration to pass
    time.sleep(rc.test_length + cfg.post_test_buffer)

    send_pkill_hosts(all_hosts, ["server", "client"])

    time.sleep(1)

    results = collect_results(cfg, client_hosts, run_dir, test_name, METRICS)
    return results, []

Lauching the test

In [49]:
import logging

logging.getLogger("paramiko").setLevel(logging.WARNING)

N_RUNS = 10

cfg = CatEvalConfig(n_runs=N_RUNS, monitor_cpu=False)
now = datetime.now().strftime("%d-%m-%H-%M%p")

test_name = f"categorization_{now}"
matrix = categorization_matrix()


def row_fields(rc):
    # columns identifying each run in the result CSVs
    return {
        "ADDITIONAL_DATA_SIZE": rc.additional_data_size,
    }


run_eval(
    matrix,
    cfg,
    run_once=run_once,
    test_name=test_name,
    row_fields=row_fields,
    metrics=METRICS,
)

=> {'ADDITIONAL_DATA_SIZE': 10000} run=0 (attempt 1)
  [cmd] done on 21/21 host(s) in 1.1s
  [server] started on 1/1 host(s) in 0.4s
  [clients] started on 20/20 host(s) in 1.0s
Started clients
-> collected {'LATENCY': 100} samples, 0 cpu samples

=> {'ADDITIONAL_DATA_SIZE': 10000} run=1 (attempt 2)
  [cmd] done on 21/21 host(s) in 1.0s
  [server] started on 1/1 host(s) in 0.4s
  [clients] started on 20/20 host(s) in 0.8s
Started clients
-> collected {'LATENCY': 100} samples, 0 cpu samples

=> {'ADDITIONAL_DATA_SIZE': 10000} run=2 (attempt 3)
  [cmd] done on 21/21 host(s) in 0.9s
  [server] started on 1/1 host(s) in 0.4s
  [clients] started on 20/20 host(s) in 1.0s
Started clients
-> collected {'LATENCY': 100} samples, 0 cpu samples

=> {'ADDITIONAL_DATA_SIZE': 10000} run=3 (attempt 4)
  [cmd] done on 21/21 host(s) in 0.9s
  [server] started on 1/1 host(s) in 0.4s
  [clients] started on 20/20 host(s) in 0.9s
Started clients
-> collected {'LATENCY': 100} samples, 0 cpu samples

=> {'ADD

{'LATENCY': PosixPath('npf-out/categorization_16-09-16-20PM.csv')}

### Downloading SQLOGs from server and relay

In [50]:
import shutil
import subprocess
from pathlib import Path

# uses cfg and test_name from the launching cell above
local_base = Path(f"./sqlogs/{test_name}")
local_base.mkdir(parents=True, exist_ok=True)

client_hosts = [
    en.Host(h.address, alias=h.alias, user="root", extra=h.extra)
    for h in experiment.roles["client"]
]

# for each client machine, download all of the qlogs from the namespaces that are stored in tmp (/tmp in g5k machines is stored on a local disk)
for client_host in client_hosts:
    host = client_host.address
    print(f"downloading sqlogs from {host}")
    subprocess.run(
        [
            "rsync",
            "-az",
            "--include=*/",
            "--include=*.sqlog",
            "--exclude=*",
            "--prune-empty-dirs",
            f"root@{host}:{cfg.remote_log_root}/{test_name}/",
            f"{local_base}/",
        ],
        check=False,
    )

# after we downloaded the files, we need to move the files up one level to remove the "qlog" dir
# ./sqlogs/{test_name}/{run_id}/client_{CLIENT_ID}
for qlog_dir in sorted(local_base.glob("*/qlog")):
    for client_dir in sorted(qlog_dir.iterdir()):
        target = qlog_dir.parent / client_dir.name
        # if the same results were already downloaded before, we replace the prev copy
        if target.exists():
            shutil.rmtree(target)
        client_dir.rename(target)
    qlog_dir.rmdir()

print(f"results: {local_base}")

downloading sqlogs from nova-4.lyon.grid5000.fr
downloading sqlogs from gros-111.nancy.grid5000.fr
downloading sqlogs from gros-109.nancy.grid5000.fr
downloading sqlogs from parasilo-28.rennes.grid5000.fr
downloading sqlogs from ecotype-46.nantes.grid5000.fr
downloading sqlogs from parasilo-18.rennes.grid5000.fr
downloading sqlogs from nova-23.lyon.grid5000.fr
downloading sqlogs from gros-110.nancy.grid5000.fr
downloading sqlogs from parasilo-14.rennes.grid5000.fr
downloading sqlogs from parasilo-19.rennes.grid5000.fr
downloading sqlogs from ecotype-34.nantes.grid5000.fr
downloading sqlogs from nova-5.lyon.grid5000.fr
downloading sqlogs from ecotype-45.nantes.grid5000.fr
downloading sqlogs from ecotype-40.nantes.grid5000.fr
downloading sqlogs from nova-8.lyon.grid5000.fr
downloading sqlogs from gros-11.nancy.grid5000.fr
downloading sqlogs from parasilo-10.rennes.grid5000.fr
downloading sqlogs from ecotype-47.nantes.grid5000.fr
downloading sqlogs from nova-10.lyon.grid5000.fr
downloadin

### Merging SQLOG files together

In [123]:
from pathlib import Path
import sys
import re
import json
import csv
import ipaddress

site_subnet_to_cluster = {
    ipaddress.ip_network(subnet, strict=False): name
    for subnet, name in zip(site_subnets_str, cluster_names)
}


def cluster_from_sqlog(sqlog: Path) -> str | None:
    match = re.search(r"client-Client-(\d+\.\d+\.\d+\.\d+)\.sqlog", sqlog.name)
    if not match:
        return None

    client_ip = ipaddress.ip_address(match.group(1))
    for subnet, cluster in site_subnet_to_cluster.items():
        if client_ip in subnet:
            return cluster

    return None


# NOTE: for the smoothed RTT extract data.smoothed_rtt from recovery:metrics_updated
def extract_smoothed_rtt(sqlog, cluster, msg_size, writer):
    with open(sqlog) as src:
        for line in src:
            line = line.strip()
            if not line:
                continue

            try:
                event = json.loads(line)
            except json.JSONDecodeError as e:
                print(e)
                # idk why so many log entries are broken
                continue

            # e.g.
            # {"time":30.884829,"name":"recovery:metrics_updated","data":{"min_rtt":15.767142,"smoothed_rtt":15.767142,"latest_rtt":15.767142,"rtt_variance":7.883571,"bytes_in_flight":0}}
            if event.get("name") == "recovery:metrics_updated":
                smoothed_rtt = event.get("data", {}).get("smoothed_rtt")
                if smoothed_rtt is not None:
                    writer.writerow(
                        [event.get("time"), cluster, msg_size, smoothed_rtt]
                    )


# to compute the download completion time, take the time from the first stream frame with stream ID 3 and the last, subtract end from start
def extract_download_completion_time(sqlog, cluster, msg_size, writer, run_index):
    with open(sqlog) as src:
        start_frame_time = None
        for line in src:
            line = line.strip()
            if not line:
                continue

            try:
                event = json.loads(line)
            except json.JSONDecodeError as e:
                print(e)
                # idk why so many log entries are broken
                continue

            # e.g.
            # Small packets: entire message contained in one QUIC packet, contains FIN flag
            #   {"time":5074.468,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":2},"raw":{"length":1085,"payload_length":1068},"frames":[{"frame_type":"unknown","raw_frame_type":246},{"frame_type":"stream","stream_id":3,"offset":0,"length":1036,"fin":true}]}}
            # large packet:
            # here is one packet that contains part of the message:
            # {"time":5072.8867,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":3},"raw":{"length":1284,"payload_length":1267},"frames":[{"frame_type":"unknown","raw_frame_type":246},{"frame_type":"stream","stream_id":3,"offset":1235,"length":1234}]}}
            # here is the final packet (with fin flag raised)
            # {"time":5307.8716,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":92},"raw":{"length":635,"payload_length":618},"frames":[{"frame_type":"unknown","raw_frame_type":246},{"frame_type":"stream","stream_id":3,"offset":99453,"length":583,"fin":true}]}}
            if event.get("name") == "transport:packet_received":
                frames = event.get("data", {}).get("frames", []) or []
                for frame in frames:
                    if (
                        frame.get("frame_type") == "stream"
                        and frame.get("stream_id") == 3
                    ):
                        fin = frame.get("fin", False)
                        # length = frame.get("length")
                        if fin:
                            end_time = event.get("time")
                            if start_frame_time is None:
                                writer.writerow(
                                    [cluster,run_index, msg_size, end_time, end_time, 0]
                                )
                                break
                            else:
                                writer.writerow(
                                    [
                                        cluster,
                                        run_index,
                                        msg_size,
                                        start_frame_time,
                                        end_time,
                                        (end_time - start_frame_time),
                                    ]
                                )
                                break
                        else:
                            if start_frame_time is None:
                                start_frame_time = event.get("time")


# Source - https://stackoverflow.com/a/16974075
# Retrieved 2026-09-16, License - CC BY-SA 3.0
def missing_elements(L):
    start, end = L[0], L[-1]
    return sorted(set(range(start, end + 1)).difference(L))

# the quiche version used does not skip packet numbers,
def extract_losses(sqlog, cluster, msg_size, writer, run_id):
    with open(sqlog) as src:
        packets_seen: dict[int, bool] = {1: True}

        for line in src:
            line = line.strip()
            if not line:
                continue

            try:
                event = json.loads(line)
            except json.JSONDecodeError as e:
                print(e)
                # idk why so many log entries are broken
                continue

            # e.g.
            # {"time":5179.9014,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":30},"raw":{"length":1284,"payload_length":1267},"frames":[{"frame_type":"unknown","raw_frame_type":246},{"frame_type":"stream","stream_id":3,"offset":31490,"length":1232}]}}
            if event.get("name") == "transport:packet_received":
                data = event.get("data", {})
                frames = data.get("frames", []) or []
                for frame in frames:
                    frame_type = frame.get("frame_type") 
                    if (
                        (frame_type == "stream" and frame.get("stream_id") == 3) or frame_type == "ping" 
                        
                    ):
                        packet_num = data.get("header").get("packet_number")
                        packets_seen[packet_num] = True


                        # if packet_num != prev_packet_num + 1:
                        #     lost = packet_num-prev_packet_num
                        #     print(
                        #         f"loss detected (run_id={run_id}?, prev_packet_num={prev_packet_num}, curr packet_num={packet_num}, diff: {lost}"
                        #     )
                        #     writer.writerow([cluster, msg_size, event.get("time"), lost])
                        #     prev_packet_num = packet_num
                        # else:
                        #     prev_packet_num = packet_num
        
        all_packet_nums = list(packets_seen.keys())
        losses = missing_elements(all_packet_nums)
        if len(losses) != 0:
            # print(f"lost packets: {losses}")
            total_lost = len(losses)
            total_sent = len(all_packet_nums)
            loss_rate = total_lost/total_sent
            writer.writerow([cluster, msg_size, total_lost, total_sent, loss_rate]) 
        


local_base = Path(f"./sqlogs/{test_name}")

trace_files = []
for run_conf in matrix:
    for run_index in range(N_RUNS):
        run_id = f"sz{run_conf.additional_data_size}_r{run_index}"

        run_dir = local_base / run_id

        # the qlogs are stored in one dir per client, and the file name contains the client's IP
        for file in sorted(run_dir.glob("client_*/client-*.sqlog")):
            # we keep the message size of the run next to each file so it can be
            # written as a column in the csvs below
            trace_files.append((file, run_index, run_conf.additional_data_size))

if not trace_files:
    print(f"no sqlog files found in {local_base}")


est_rtt_csv = Path(f"./npf-out/") / test_name / f"est_rtt.csv"
dl_completion_csv = Path(f"./npf-out/") / test_name / f"dl_completion.csv"
losses_csv = Path(f"./npf-out/") / test_name / f"losses.csv"
est_rtt_csv.parent.mkdir(parents=True, exist_ok=True)

# the files are parsed one by one so each row can be tagged with the cluster of its client
with (
    open(est_rtt_csv, "w", newline="") as rtt_out,
    open(dl_completion_csv, "w", newline="") as dl_out,
    open(losses_csv, "w", newline="") as losses,
):
    rtt_writer = csv.writer(rtt_out)
    dl_writer = csv.writer(dl_out)
    losses_writer = csv.writer(losses)
    rtt_writer.writerow(["time", "cluster", "msg_size", "smoothed_rtt"])
    dl_writer.writerow(["cluster", "run_index", "msg_size", "start", "end", "duration"])
    losses_writer.writerow(["cluster", "msg_size", "lost", "total_msg_packets", "loss_rate"])

    for file, run_index, msg_size in trace_files:
        cluster = cluster_from_sqlog(file)
        if cluster is None:
            print(f"couldn't find the cluster of {file}, skipping")
            continue
        extract_smoothed_rtt(file, cluster, msg_size, rtt_writer)
        extract_download_completion_time(file, cluster, msg_size, dl_writer, run_index)
        extract_losses(file, cluster, msg_size, losses_writer, f"sz{msg_size}_r{run_index}")

print(f"estimated RTT csv: {est_rtt_csv}")
print(f"download completion csv: {dl_completion_csv}")
print(f"losses csv: {losses_csv}")

estimated RTT csv: npf-out/categorization_16-09-16-20PM/est_rtt.csv
download completion csv: npf-out/categorization_16-09-16-20PM/dl_completion.csv
losses csv: npf-out/categorization_16-09-16-20PM/losses.csv


### Graphing the results


In [127]:
import subprocess
from pathlib import Path

NO_TITLE = True
out_path = f"./graphs/{test_name}"
output_path = Path(out_path)
output_path.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        "./mcast_graphs.py",
        f"./npf-out/raw/{test_name}",  # raw client logs (they contain the per-cluster RESULT-LATENCY lines)
        out_path,  # out path
        test_name,
        # one cdf graph is plotted for each additional data size, you can restrict it with:
        # "--data-size", "1000",
        f"./npf-out/{test_name}/est_rtt.csv",  # estimated rtt csv
        f"./npf-out/{test_name}/dl_completion.csv",  # download completion time csv
        f"./npf-out/{test_name}/losses.csv",  # losses csv
        *(
            ["--no-title"] if NO_TITLE else []
        ),  # list unpacking, this avoids the empty ""
    ],
    check=True,
)

wrote ./graphs/categorization_16-09-16-20PM/rct_vs_runs_normalized.svg
wrote ./graphs/categorization_16-09-16-20PM/rct_vs_runs.svg
wrote ./graphs/categorization_16-09-16-20PM/rct_vs_cluster_normalized.svg
wrote ./graphs/categorization_16-09-16-20PM/rct_vs_cluster.svg
wrote ./graphs/categorization_16-09-16-20PM/losses_vs_cluster.svg


CompletedProcess(args=['./mcast_graphs.py', './npf-out/raw/categorization_16-09-16-20PM', './graphs/categorization_16-09-16-20PM', 'categorization_16-09-16-20PM', './npf-out/categorization_16-09-16-20PM/est_rtt.csv', './npf-out/categorization_16-09-16-20PM/dl_completion.csv', './npf-out/categorization_16-09-16-20PM/losses.csv', '--no-title'], returncode=0)

#### Compressing the csv results

In [ ]:
import subprocess

# compress all related files in one tarball
subprocess.run(
    [
        "tar",
        "czf",
        f"./npf-out/all_{test_name}.tar.gz",
        f"./npf-out/{test_name}.csv",
        f"./npf-out/{test_name}_cpu.csv",
        f"./npf-out/ack_rate_{test_name}/",
        f"./npf-out/raw/{test_name}/",
        f"./sqlogs/{test_name}/",
    ],
    check=True,
)

# move archive to the graph dir of the test
subprocess.run(
    [
        "mv",
        f"./npf-out/all_{test_name}.tar.gz",
        f"./graphs/{test_name}/{test_name}.tar.gz",
    ],
    check=True,
)

# delete the csvs and directories
subprocess.run(
    [
        "rm",
        f"./npf-out/{test_name}.csv",
        f"./npf-out/{test_name}_cpu.csv",
    ],
    check=True,
)
subprocess.run(
    [
        "rm",
        "-rf",
        f"./npf-out/ack_rate_{test_name}/",
        f"./npf-out/raw/{test_name}/",
        f"./sqlogs/{test_name}/",
    ],
    check=True,
)

Opposite code to unarchive the results, in order to regenerate graphs if needed

In [ ]:
import subprocess
from pathlib import Path

# og_name = "sserv_2thr_latency_test_large_relay_topo_26-04-21-39PM_481mbps"
test_name = "serv_2thr_latency_test_large_relay_topo_26-04-21-39PM"

archive_path = Path(f"./graphs/{test_name}/{test_name}.tar.gz")

if archive_path.exists():
    print(f"decompressing {archive_path}...")

    Path("./npf-out/").mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            "tar",
            "xzf",
            str(archive_path),
            "-C",
            "./",
        ],
        check=True,
    )
    print(f"decompressed files to ./npf-out/ and ./sqlogs/")
else:
    print(f"Couldn't find: {archive_path}")

#### Deleting log files from all clusters

In [ ]:
import subprocess
from pathlib import Path

remote_log_root = "/tmp/logs"
matrix = categorization_matrix()
for run_conf in matrix:

    remote_qlog_dir = f"{remote_log_root}/{test_name}/"

    en.run_command(
        f"rm -rf {remote_qlog_dir}",
        roles=experiment.roles["client"] + experiment.roles["server"],
    )

print("done deleting sqlog files")

## Important: Stopping the current booking
Always, always stop your booking if you are done earlier.

In [ ]:
experiment.stop_reservation()